In [1]:
# Cell 1: 필수 라이브러리 설치
%pip install imagehash pillow requests --quiet


Note: you may need to restart the kernel to use updated packages.


In [9]:
# Cell 2: 라이브러리 import 및 설정
import os
import json
import imagehash
from PIL import Image
import requests
from io import BytesIO
from difflib import SequenceMatcher
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIError, APIConnectionError, AuthenticationError

# 설정
SIMILARITY_THRESHOLD = 0.4  # 유사도 임계값 (0.4 이상인 것만 Vision API로 비교)
IMAGE_WEIGHT = 0.35  # 이미지 유사도 가중치
PRICE_WEIGHT = 0.15  # 가격 유사도 가중치
TITLE_WEIGHT = 0.25  # 상품명 유사도 가중치
OCR_SPECS_WEIGHT = 0.25  # 상품 상세 정보 유사도 가중치
MAX_WORKERS = 10  # 병렬 처리 스레드 수
TOTAL_WEIGHT = IMAGE_WEIGHT + PRICE_WEIGHT + TITLE_WEIGHT + OCR_SPECS_WEIGHT

VISION_MODEL = "gpt-4o-mini"
VISION_DETAIL = "low"
VISION_TOP_K = 3
VISION_MAX_RETRIES = 3
VISION_BASE_WAIT = 2
VISION_RATE_LIMIT_WAIT = 10

print("✅ 라이브러리 import 완료")
print(f"📊 유사도 임계값: {SIMILARITY_THRESHOLD}")
print(f"⚖️ 가중치 - 이미지: {IMAGE_WEIGHT}, 가격: {PRICE_WEIGHT}, 상품명: {TITLE_WEIGHT}, 상세정보: {OCR_SPECS_WEIGHT}")
print(f"⚡ 병렬 처리: 최대 {MAX_WORKERS}개 스레드")
print(f"👁️ Vision 모델: {VISION_MODEL} (detail={VISION_DETAIL}, TOP_K={VISION_TOP_K})\n")


✅ 라이브러리 import 완료
📊 유사도 임계값: 0.4
⚖️ 가중치 - 이미지: 0.35, 가격: 0.15, 상품명: 0.25, 상세정보: 0.25
⚡ 병렬 처리: 최대 10개 스레드
👁️ Vision 모델: gpt-4o-mini (detail=low, TOP_K=3)



In [10]:
# Cell 3: 이미지 해싱 함수 (캐싱 포함)

# 전역 캐시 딕셔너리
_image_hash_cache = {}

def get_image_hash(image_url, timeout=3):
    """
    이미지 URL에서 perceptual hash를 계산합니다 (캐싱 포함).
    
    Args:
        image_url: 이미지 URL
        timeout: 요청 타임아웃 (초)
    
    Returns:
        imagehash.ImageHash 또는 None (실패 시)
    """
    # 캐시 확인
    if image_url in _image_hash_cache:
        return _image_hash_cache[image_url]
    
    try:
        response = requests.get(image_url, timeout=timeout)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        hash_value = imagehash.phash(img)  # perceptual hash 사용
        
        # 캐시에 저장
        _image_hash_cache[image_url] = hash_value
        return hash_value
    except Exception:
        # 실패한 경우 캐시에 저장하지 않아 일시적 오류 시 재시도 가능
        return None


def calculate_image_similarity(url1, url2):
    """
    두 이미지의 유사도를 계산합니다 (0-1, 1에 가까울수록 유사).
    
    Args:
        url1: 첫 번째 이미지 URL
        url2: 두 번째 이미지 URL
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    hash1 = get_image_hash(url1)
    hash2 = get_image_hash(url2)
    
    if hash1 is None or hash2 is None:
        return 0.0
    
    # Hamming distance 계산 (0-64)
    difference = hash1 - hash2
    
    # 유사도로 변환 (0-1)
    similarity = 1 - (difference / 64.0)
    return max(0.0, min(1.0, similarity))  # 0-1 범위로 제한


print("✅ 이미지 해싱 함수 정의 완료")
print("   - get_image_hash(): 이미지 URL에서 해시 계산")
print("   - calculate_image_similarity(): 두 이미지의 유사도 계산\n")


✅ 이미지 해싱 함수 정의 완료
   - get_image_hash(): 이미지 URL에서 해시 계산
   - calculate_image_similarity(): 두 이미지의 유사도 계산



In [11]:
# Cell 4: 상품명 유사도 계산 함수

def remove_marketing_words(text):
    """
    상품명에서 마케팅 문구를 제거합니다.
    
    Args:
        text: 원본 텍스트
    
    Returns:
        str: 마케팅 문구가 제거된 텍스트
    """
    if not text:
        return ""
    
    # 마케팅 문구 목록
    marketing_words = [
        "최신형", "신상품", "신제품", "신규", "신작",
        "특가", "할인", "세일", "프로모션", "이벤트",
        "무료배송", "로켓배송", "당일배송", "빠른배송",
        "인기", "추천", "베스트", "핫딜", "초특가",
        "프리미엄", "고급형", "럭셔리", "프리미엄",
        "무료", "증정", "사은품", "혜택", "쿠폰",
        "할인가", "특가가", "세일가", "할인가격",
        "지금", "오늘", "오늘만", "한정", "한정수량",
        "품절임박", "재고부족", "마감임박", "서둘러",
        "ad", "advertisement", "광고"
    ]
    
    cleaned_text = text.lower()
    
    # 마케팅 문구 제거
    for word in marketing_words:
        cleaned_text = cleaned_text.replace(word.lower(), " ")
    
    # 연속된 공백을 하나로 통합
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text


def extract_spec_keywords(text):
    """
    텍스트에서 제품 사양 키워드를 추출합니다 (범용 패턴).
    모든 제품 카테고리에 적용 가능한 패턴을 사용합니다.
    
    Args:
        text: 원본 텍스트
    
    Returns:
        set: 추출된 키워드 집합
    """
    if not text:
        return set()
    
    keywords = set()
    
    # 1. 용량/크기 패턴 (범용)
    # 메모리/저장용량: GB, TB, MB
    capacity_patterns = [
        r'\d+\s*GB',  # 예: 16GB, 16 GB, 512GB
        r'\d+\s*TB',  # 예: 1TB, 2 TB
        r'\d+\s*MB',  # 예: 128MB
    ]
    
    # 무게: kg, g
    weight_patterns = [
        r'\d+\.?\d*\s*kg',  # 예: 1.5kg, 2kg, 1.5 kg
        r'\d+\s*g\b',  # 예: 500g, 1000g (단어 경계로 'g'만 매칭)
    ]
    
    # 부피/용량: ml, l
    volume_patterns = [
        r'\d+\s*ml',  # 예: 500ml, 1000 ml
        r'\d+\s*l\b',  # 예: 1l, 2 l (단어 경계)
    ]
    
    # 길이/크기: cm, mm, inch, 인치
    size_patterns = [
        r'\d+\.?\d*\s*cm',  # 예: 15.6cm, 30 cm
        r'\d+\.?\d*\s*mm',  # 예: 8.6mm, 10 mm
        r'\d+\.?\d*\s*inch',  # 예: 15.6inch, 10 inch
        r'\d+\.?\d*\s*인치',  # 예: 15.6인치, 10인치
    ]
    
    # 2. 치수 패턴 (범용)
    # 가로x세로, 해상도 등
    dimension_patterns = [
        r'\d+\s*x\s*\d+',  # 예: 1920x1080, 30x40, 100 x 200
        r'\d+\s*X\s*\d+',  # 대문자 X
    ]
    
    # 3. 모델 번호 패턴 (범용)
    # 알파벳+숫자 조합 (예: iPhone14, GalaxyS23, NikeAir, i5-12400)
    model_patterns = [
        r'[A-Za-z]+\s*\d+[-\s]?\d*',  # 예: iPhone14, GalaxyS23, i5-12400, RTX3060
        r'[A-Za-z]+\s*[A-Za-z]+\s*\d+',  # 예: Nike Air 1, Samsung Galaxy 23
    ]
    
    # 4. 컴퓨터/전자제품 전용 패턴 (유지)
    computer_patterns = [
        r'i\d+[-\s]?\d+',  # CPU 모델 (예: i5-12400, i5 12400, i7-13700)
        r'RTX\s*\d+',  # GPU 모델 (예: RTX 3060, RTX4060)
        r'GTX\s*\d+',  # GPU 모델 (예: GTX 1660)
        r'SSD\s*\d+',  # SSD 용량 (예: SSD 512)
        r'\d+\s*세대',  # CPU 세대 (예: 13세대, 14세대)
    ]
    
    # 5. 색상 패턴 (범용)
    # 일반적인 색상 키워드
    color_keywords = [
        '블랙', '화이트', '실버', '골드', '로즈골드',
        '레드', '블루', '그린', '옐로우', '퍼플',
        '핑크', '그레이', '베이지', '브라운', '네이비',
        '아이보리', '오렌지', '민트', '카키', '버건디',
        'black', 'white', 'silver', 'gold', 'red', 'blue', 'green', 'pink', 'gray', 'brown'
    ]
    
    # 6. 수량/개수 패턴 (범용)
    # 예: 1개, 2개입, 3개세트, 10EA, 1SET
    quantity_patterns = [
        r'\d+\s*개',  # 예: 1개, 2개입, 3개세트
        r'\d+\s*EA',  # 예: 10EA, 1EA
        r'\d+\s*SET',  # 예: 1SET, 2SET
        r'\d+\s*세트',  # 예: 1세트, 2세트
        r'\d+\s*팩',  # 예: 1팩, 10팩
        r'\d+\s*입',  # 예: 2입, 3입
        r'\d+\s*매',  # 예: 10매, 100매
        r'\d+\s*장',  # 예: 10장, 100장
        r'\d+\s*병',  # 예: 1병, 6병
        r'\d+\s*봉',  # 예: 1봉, 10봉
        r'\d+\s*박스',  # 예: 1박스
        r'\d+\s*box',  # 예: 1box
    ]
    
    # 7. 재질/소재 키워드 (범용)
    # 의류, 가구, 식품 포장 등 다양한 제품에 적용
    material_keywords = [
        # 의류/직물
        '면', '폴리에스터', '나일론', '레이온', '린넨', '울', '캐시미어',
        '데님', '코튼', '실크', '새틴', '벨벳', '니트', '스웨이드', '모달',
        '스판', '폴리', '아크릴', '비스코스', '텐셀',
        # 가구/인테리어
        '나무', '원목', '합판', 'MDF', '파티클보드', '강화유리', '유리',
        '스테인리스', '스틸', '알루미늄', '철', '철제', '플라스틱', 'PVC',
        '가죽', '인조가죽', 'PU', '천', '원단', '패브릭',
        # 식품/포장
        'PET', 'PP', 'PE', 'PS', '종이', '스티로폼',
        # 기타
        '세라믹', '도자기', '고무', '실리콘', '메탈', '크롬', '황동', '구리',
        # 영어
        'cotton', 'polyester', 'leather', 'wood', 'metal', 'plastic', 'glass', 'ceramic', 'stainless'
    ]
    
    # 8. 원산지/국가 키워드 (범용)
    origin_keywords = [
        '한국', '국산', '국내산', '중국', '미국', '일본', '독일', '이탈리아',
        '베트남', '태국', '인도네시아', '필리핀', '인도', '대만', '프랑스', '스페인',
        'korea', 'china', 'usa', 'japan', 'germany', 'italy', 'vietnam', 'france',
        'made in', '제조국', '원산지', '수입', '직수입'
    ]
    
    # 9. 인증/표준 패턴 (범용)
    # 예: KC인증, CE인증, FDA승인, ISO9001
    certification_patterns = [
        r'KC\s*인증',  # 예: KC인증
        r'CE\s*인증',  # 예: CE인증
        r'FDA\s*승인',  # 예: FDA승인
        r'ISO\s*\d+',  # 예: ISO9001, ISO14001
        r'KS\s*인증',  # 예: KS인증
        r'HACCP',  # 식품 안전 인증
        r'GMP',  # 의약품 제조 기준
        r'UL\s*인증',  # 예: UL인증
        r'친환경',  # 친환경 인증
        r'유기농',  # 유기농 인증
        r'무농약',  # 무농약 인증
    ]
    
    # 10. 등급/품질 패턴 (범용)
    # 예: 1등급, A급, 프리미엄, 특1급
    grade_patterns = [
        r'\d+\s*등급',  # 예: 1등급, 2등급
        r'[A-F]\s*급',  # 예: A급, B급
        r'특\s*\d+\s*급',  # 예: 특1급, 특2급
        r'\d+\+\+?',  # 예: 1+, 1++
    ]
    grade_keywords = [
        '프리미엄', '고급', '일반', '표준', '특급', '최상급',
        'premium', 'grade', 'quality', '명품'
    ]
    
    # 11. 기능/특성 키워드 (범용)
    # 예: 방수, 방진, 내열, 내구성
    feature_keywords = [
        '방수', '방진', '내열', '내구', '방오', '방취', '항균',
        '방충', '방습', '통기', '흡수', '흡습', '속건', '보온', '냉감',
        '무선', '유선', '충전식', '건전지', '자동', '수동',
        'waterproof', 'dustproof', 'heatproof', 'durable', 'wireless', 'auto'
    ]
    
    # 12. 날짜/기한 패턴 (범용)
    # 예: 2025-12-31, 2025.12.31, 유통기한, 제조일자
    date_patterns = [
        r'\d{4}[-\s./]\d{1,2}[-\s./]\d{1,2}',  # 예: 2025-12-31, 2025.12.31
    ]
    date_keywords = [
        '유통기한', '제조일자', '제조일', '소비기한', '신선', '당일'
    ]
    
    # 13. 사이즈/의류 패턴 (범용)
    size_keywords = [
        'XS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL', '2XL', '3XL', '4XL',
        'FREE', '프리사이즈', '단일사이즈',
        '소', '중', '대', '특대',
        '85', '90', '95', '100', '105', '110',  # 한국 의류 사이즈
        '44', '55', '66', '77', '88',  # 여성복 사이즈
    ]
    
    # 모든 정규식 패턴 적용
    all_patterns = (
        capacity_patterns + weight_patterns + volume_patterns + 
        size_patterns + dimension_patterns + model_patterns + computer_patterns +
        quantity_patterns + certification_patterns + grade_patterns + date_patterns
    )
    
    for pattern in all_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            # 공백 제거 및 정규화
            normalized = re.sub(r'\s+', '', str(match).upper())
            if normalized and len(normalized) > 1:  # 최소 2글자 이상
                keywords.add(normalized)
    
    # 키워드 리스트 매칭 (대소문자 무시)
    text_lower = text.lower()
    
    # 색상 키워드
    for color in color_keywords:
        if color.lower() in text_lower:
            keywords.add(color.upper())
    
    # 재질/소재 키워드
    for material in material_keywords:
        if material.lower() in text_lower:
            keywords.add(material.upper())
    
    # 원산지 키워드
    for origin in origin_keywords:
        if origin.lower() in text_lower:
            keywords.add(origin.upper())
    
    # 등급 키워드
    for grade in grade_keywords:
        if grade.lower() in text_lower:
            keywords.add(grade.upper())
    
    # 기능/특성 키워드
    for feature in feature_keywords:
        if feature.lower() in text_lower:
            keywords.add(feature.upper())
    
    # 날짜 키워드
    for date_kw in date_keywords:
        if date_kw.lower() in text_lower:
            keywords.add(date_kw.upper())
    
    # 사이즈 키워드
    for size in size_keywords:
        if size.lower() in text_lower:
            keywords.add(size.upper())
    
    return keywords


def calculate_title_similarity(title1, title2):
    """
    두 상품명의 유사도를 계산합니다 (단어 기반, 0-1, 1에 가까울수록 유사).
    
    Args:
        title1: 첫 번째 상품명
        title2: 두 번째 상품명
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    if not title1 or not title2:
        return 0.0
    
    # 1. 마케팅 문구 제거
    title1_cleaned = remove_marketing_words(title1)
    title2_cleaned = remove_marketing_words(title2)
    
    if not title1_cleaned or not title2_cleaned:
        return 0.0
    
    # 2. 단어로 분리
    words1 = set(word for word in title1_cleaned.split() if len(word) > 1)  # 1글자 단어 제외
    words2 = set(word for word in title2_cleaned.split() if len(word) > 1)
    
    if not words1 or not words2:
        return 0.0
    
    # 3. Jaccard 유사도 (단어 집합 기반) - 단어 순서 무시
    intersection = words1 & words2
    union = words1 | words2
    jaccard_sim = len(intersection) / len(union) if union else 0.0
    
    # 4. SequenceMatcher 유사도 (기존 방식 유지 - 부분 일치 고려)
    sequence_sim = SequenceMatcher(None, title1_cleaned, title2_cleaned).ratio()
    
    # 5. 제품 사양 키워드 매칭 (중요한 키워드에 가중치)
    spec_keywords1 = extract_spec_keywords(title1)
    spec_keywords2 = extract_spec_keywords(title2)
    
    spec_sim = 0.0
    if spec_keywords1 or spec_keywords2:
        spec_common = spec_keywords1 & spec_keywords2
        spec_total = spec_keywords1 | spec_keywords2
        spec_sim = len(spec_common) / len(spec_total) if spec_total else 0.0
    
    # 6. 가중 평균 (단어 50%, 시퀀스 30%, 키워드 20%)
    # 키워드가 없는 경우 단어 60%, 시퀀스 40%
    if spec_sim > 0:
        combined = (jaccard_sim * 0.5) + (sequence_sim * 0.3) + (spec_sim * 0.2)
    else:
        combined = (jaccard_sim * 0.6) + (sequence_sim * 0.4)
    
    return max(0.0, min(1.0, combined))


print("✅ 상품명 유사도 계산 함수 정의 완료")
print("   - remove_marketing_words(): 마케팅 문구 제거")
print("   - extract_spec_keywords(): 제품 사양 키워드 추출")
print("   - calculate_title_similarity(): 두 상품명의 유사도 계산 (단어 기반)\n")


✅ 상품명 유사도 계산 함수 정의 완료
   - remove_marketing_words(): 마케팅 문구 제거
   - extract_spec_keywords(): 제품 사양 키워드 추출
   - calculate_title_similarity(): 두 상품명의 유사도 계산 (단어 기반)



In [12]:
# Cell 5: 가격 유사도 계산 함수

def parse_price(price_value):
    """문자열/숫자 형태의 가격을 float로 변환합니다."""
    if price_value is None:
        return None
    if isinstance(price_value, (int, float)):
        return float(price_value)
    cleaned = re.sub(r"[^0-9.]", "", str(price_value))
    if not cleaned:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def calculate_price_similarity(price1, price2):
    """두 가격의 유사도(0-1)를 계산합니다."""
    p1 = parse_price(price1)
    p2 = parse_price(price2)

    if p1 is None or p2 is None or max(p1, p2) == 0:
        return 0.0

    diff_ratio = abs(p1 - p2) / max(p1, p2)
    return max(0.0, 1 - diff_ratio)


print("✅ 가격 유사도 계산 함수 정의 완료")
print("   - parse_price(): 가격 문자열을 float로 변환")
print("   - calculate_price_similarity(): 두 가격의 유사도 계산\n")


✅ 가격 유사도 계산 함수 정의 완료
   - parse_price(): 가격 문자열을 float로 변환
   - calculate_price_similarity(): 두 가격의 유사도 계산



In [13]:
# Cell 6: 상품 상세 정보 유사도 계산 함수

def normalize_keyword(keyword):
    """
    키워드를 정규화하여 비교 가능하게 만듭니다.
    
    Args:
        keyword: 원본 키워드
    
    Returns:
        str: 정규화된 키워드
    """
    if not keyword:
        return ""
    normalized = keyword.lower().strip()
    # 공백, 하이픈, 언더스코어 제거
    normalized = re.sub(r'[\s\-_]+', '', normalized)
    return normalized


def convert_specs_to_text(detail_specs):
    """
    싸다구의 구조화된 상세 정보를 텍스트로 변환합니다.
    중요한 사양만 추출하여 노이즈를 줄입니다.
    
    Args:
        detail_specs: 싸다구 상세 정보 딕셔너리
    
    Returns:
        str: 변환된 텍스트
    """
    if not detail_specs or not isinstance(detail_specs, dict):
        return ""
    
    # 중요한 사양 키만 추출 (모든 제품 카테고리에 적용 가능)
    important_keys = [
        # 공통 (모든 제품)
        "상표", "브랜드", "모델", "모델명", "제조사", "제조국", "원산지",
        "색상", "색깔", "컬러", "무게", "중량", "제품 크기", "크기", "사이즈", "규격",
        
        # 컴퓨터/전자제품
        "CPU 유형", "프로세서", "CPU 주파수", "프로세서 주파수", "프로세서 코어", "CPU 주요 주파수",
        "메모리 용량", "RAM", "하드 드라이브 용량", "저장 유형", "하드 디스크 용량", "SSD", "HDD",
        "화면 크기", "해결책", "해상도", "그래픽 카드", "그래픽 카드 유형", "비디오 메모리 용량", "GPU",
        "운영 체제", "OS", "배터리", "전압", "와트", "전력",
        
        # 의류/패션
        "소재", "재질", "원단", "혼용률", "세탁방법", "시즌", "핏",
        
        # 식품/음료
        "용량", "내용량", "유통기한", "소비기한", "제조일자", "칼로리", "열량",
        "성분", "원재료", "알레르기", "보관방법", "영양정보",
        
        # 가구/인테리어
        "재료", "프레임", "쿠션", "높이", "너비", "깊이", "두께",
        
        # 화장품/뷰티
        "피부타입", "용도", "효능", "성분", "향",
        
        # 인증/등급
        "인증", "등급", "에너지등급"
    ]
    
    text_parts = []
    for key in important_keys:
        if key in detail_specs:
            value = str(detail_specs[key]).strip()
            # 노이즈 값 제거
            noise_values = [
                "선택 사항", "기계를 보세요", "N/A", "", "포함되지 않습니다", "포함하지 않음",
                "기타", "다른", "다른 크기", "다른 저장 유형", "다른 프로세서", "기타 프로세서",
                "기타 주요 주파수", "다른 주요 주파수", "기타 운영 체제", "다른 운영 체제"
            ]
            if value and value not in noise_values:
                text_parts.append(f"{key} {value}")
    
    return " ".join(text_parts)


def clean_ocr_text(ocr_text):
    """
    OCR 텍스트에서 노이즈를 제거하고 중요한 사양 정보만 추출합니다.
    
    Args:
        ocr_text: 원본 OCR 텍스트
    
    Returns:
        str: 정제된 텍스트
    """
    if not ocr_text:
        return ""
    
    # 마케팅 문구 제거
    marketing_patterns = [
        r'배송.*?|도착.*?|출고.*?|당일.*?',
        r'특가|할인|프로모션|이벤트|증정|사은품',
        r'고객센터.*?|문의.*?|카카오톡.*?',
        r'AS.*?|보증.*?|무상.*?',
        r'게이밍.*?RGB.*?|LED.*?',
        r'PUBG|BATTLEGROUNDS|GTA5|OVERWATCH|LOSTARK|BLACKDESERT|FIFA',
        r'배틀그라운드|오버워치|로스트아크|블랙데저트|피파',
    ]
    
    cleaned = ocr_text
    for pattern in marketing_patterns:
        cleaned = re.sub(pattern, ' ', cleaned, flags=re.IGNORECASE)
    
    # 연속된 공백 제거
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    return cleaned


def calculate_ocr_specs_similarity(ocr_text, detail_specs):
    """
    쿠팡 OCR 텍스트와 싸다구 상세 정보의 유사도를 계산합니다 (개선된 하이브리드 접근).
    
    Args:
        ocr_text: 쿠팡 OCR 텍스트 (문자열)
        detail_specs: 싸다구 상세 정보 (딕셔너리)
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    # 1. 싸다구 상세 정보를 텍스트로 변환
    specs_text = convert_specs_to_text(detail_specs)
    
    if not ocr_text or not specs_text:
        return 0.0
    
    # 2. OCR 텍스트 정제 (노이즈 제거)
    cleaned_ocr = clean_ocr_text(ocr_text)
    
    # 3. 키워드 매칭 (제품 사양 키워드 추출) - 우선 수행
    ocr_keywords = extract_spec_keywords(cleaned_ocr)
    specs_keywords = extract_spec_keywords(specs_text)
    
    keyword_sim = 0.0
    if ocr_keywords or specs_keywords:
        # 정규화된 키워드 세트
        normalized_ocr = {normalize_keyword(k) for k in ocr_keywords}
        normalized_specs = {normalize_keyword(k) for k in specs_keywords}
        
        # 정확히 일치하는 키워드
        exact_matches = normalized_ocr & normalized_specs
        
        # 부분 매칭 (한 키워드가 다른 키워드에 포함)
        partial_matches = 0
        remaining_ocr = normalized_ocr - exact_matches
        remaining_specs = normalized_specs - exact_matches
        
        for k1 in list(remaining_ocr):
            for k2 in list(remaining_specs):
                if k1 in k2 or k2 in k1:
                    partial_matches += 1
                    remaining_ocr.discard(k1)
                    remaining_specs.discard(k2)
                    break
        
        # 유사도 기반 매칭 (fuzzy matching) - 임계값 낮춤
        fuzzy_matches = 0
        for k1 in list(remaining_ocr):
            best_sim = 0.0
            best_match = None
            
            for k2 in list(remaining_specs):
                sim = SequenceMatcher(None, k1, k2).ratio()
                if sim > 0.65 and sim > best_sim:  # 65% 이상 유사하면 매칭 (75% → 65%로 완화)
                    best_match = k2
                    best_sim = sim
            
            if best_match:
                fuzzy_matches += 1
                remaining_ocr.discard(k1)
                remaining_specs.discard(best_match)
        
        # Jaccard 유사도 기반 계산 (개선)
        # 정확 일치: 1.0, 부분 매칭: 0.8, 유사 매칭: 0.6
        matched_keywords = len(exact_matches) + (partial_matches * 0.8) + (fuzzy_matches * 0.6)
        total_unique = len(normalized_ocr | normalized_specs)
        
        if total_unique > 0:
            # Jaccard 유사도: 교집합 / 합집합
            keyword_sim = matched_keywords / total_unique
        else:
            keyword_sim = 0.0
        
        keyword_sim = min(1.0, keyword_sim)
    
    # 4. 텍스트 유사도 계산 (키워드가 포함된 부분만 비교)
    # 키워드가 있는 경우, 키워드 주변 텍스트만 추출하여 비교
    if ocr_keywords and specs_keywords:
        # 키워드 주변 텍스트 추출 (각 키워드 앞뒤 20자)
        ocr_keyword_contexts = []
        for keyword in ocr_keywords:
            keyword_lower = keyword.lower()
            idx = cleaned_ocr.lower().find(keyword_lower)
            if idx != -1:
                start = max(0, idx - 20)
                end = min(len(cleaned_ocr), idx + len(keyword) + 20)
                ocr_keyword_contexts.append(cleaned_ocr[start:end])
        
        specs_keyword_contexts = []
        for keyword in specs_keywords:
            keyword_lower = keyword.lower()
            idx = specs_text.lower().find(keyword_lower)
            if idx != -1:
                start = max(0, idx - 20)
                end = min(len(specs_text), idx + len(keyword) + 20)
                specs_keyword_contexts.append(specs_text[start:end])
        
        # 키워드 컨텍스트 유사도 계산
        context_sim = 0.0
        if ocr_keyword_contexts and specs_keyword_contexts:
            context_text_ocr = ' '.join(ocr_keyword_contexts)
            context_text_specs = ' '.join(specs_keyword_contexts)
            context_sim = SequenceMatcher(None, context_text_ocr.lower(), context_text_specs.lower()).ratio()
        
        # 전체 텍스트 유사도도 계산 (가중치 낮춤)
        full_text_sim = SequenceMatcher(None, cleaned_ocr.lower(), specs_text.lower()).ratio()
        
        # 컨텍스트 유사도와 전체 텍스트 유사도 결합
        text_sim = (context_sim * 0.7) + (full_text_sim * 0.3)
    else:
        # 키워드가 없는 경우 전체 텍스트 유사도만 사용
        text_sim = SequenceMatcher(None, cleaned_ocr.lower(), specs_text.lower()).ratio()
    
    # 5. 최종 가중 평균 (키워드 60%, 텍스트 40%)
    # 키워드 매칭이 더 중요하므로 가중치 증가
    if keyword_sim > 0:
        combined = (keyword_sim * 0.6) + (text_sim * 0.4)
    else:
        combined = text_sim * 0.7  # 키워드가 없으면 텍스트 유사도에 페널티
    
    return max(0.0, min(1.0, combined))


print("✅ 상품 상세 정보 유사도 계산 함수 정의 완료")
print("   - normalize_keyword(): 키워드 정규화")
print("   - convert_specs_to_text(): 구조화된 상세 정보를 텍스트로 변환")
print("   - clean_ocr_text(): OCR 텍스트 노이즈 제거")
print("   - calculate_ocr_specs_similarity(): OCR 텍스트와 상세 정보의 유사도 계산 (개선된 하이브리드)\n")


✅ 상품 상세 정보 유사도 계산 함수 정의 완료
   - normalize_keyword(): 키워드 정규화
   - convert_specs_to_text(): 구조화된 상세 정보를 텍스트로 변환
   - clean_ocr_text(): OCR 텍스트 노이즈 제거
   - calculate_ocr_specs_similarity(): OCR 텍스트와 상세 정보의 유사도 계산 (개선된 하이브리드)



In [14]:
# Cell 7: 종합 유사도 계산 및 필터링 함수

def calculate_combined_similarity(product1, product2):
    """
    이미지, 가격, 상품명, 상세정보 유사도를 종합하여 계산합니다.
    
    Args:
        product1: 첫 번째 상품 정보 (dict) - 쿠팡 상품
        product2: 두 번째 상품 정보 (dict) - 싸다구 상품
    
    Returns:
        dict: {
            "combined_similarity": 종합 유사도 (0-1),
            "image_similarity": 이미지 유사도 (0-1),
            "price_similarity": 가격 유사도 (0-1),
            "title_similarity": 상품명 유사도 (0-1),
            "ocr_specs_similarity": 상세정보 유사도 (0-1)
        }
    """
    # 이미지 유사도 계산
    img_url1 = product1.get("thumbnail_url", "")
    img_url2 = product2.get("thumbnail_url", "")
    image_sim = calculate_image_similarity(img_url1, img_url2) if img_url1 and img_url2 else 0.0
    
    # 상품명 유사도 계산
    title1 = product1.get("title", "")
    title2 = product2.get("title", "")
    title_sim = calculate_title_similarity(title1, title2)
    
    # 가격 유사도 계산 (표시가 우선, 없으면 원가)
    price1 = product1.get("displayed_price") or product1.get("price") or product1.get("original_price")
    price2 = product2.get("displayed_price") or product2.get("price") or product2.get("original_price")
    price_sim = calculate_price_similarity(price1, price2)
    
    # 상세 정보 유사도 계산 (쿠팡 OCR 텍스트 vs 싸다구 상세 정보)
    ocr_text = product1.get("ocr_text", "")
    detail_specs = product2.get("detail_specs", {})
    ocr_specs_sim = calculate_ocr_specs_similarity(ocr_text, detail_specs) if ocr_text and detail_specs else 0.0
    
    # 가중 평균으로 종합 유사도 계산
    weighted_sum = (image_sim * IMAGE_WEIGHT) + (price_sim * PRICE_WEIGHT) + (title_sim * TITLE_WEIGHT) + (ocr_specs_sim * OCR_SPECS_WEIGHT)
    combined = weighted_sum / TOTAL_WEIGHT if TOTAL_WEIGHT else 0.0
    
    return {
        "combined_similarity": combined,
        "image_similarity": image_sim,
        "price_similarity": price_sim,
        "title_similarity": title_sim,
        "ocr_specs_similarity": ocr_specs_sim
    }


def filter_similar_products(products_coupang, products_ssadagu, threshold=SIMILARITY_THRESHOLD, max_workers=MAX_WORKERS):
    """
    쿠팡과 싸다구 상품 중 유사한 상품 쌍을 필터링합니다 (병렬 처리).
    
    Args:
        products_coupang: 쿠팡 상품 리스트
        products_ssadagu: 싸다구 상품 리스트
        threshold: 유사도 임계값 (기본값: SIMILARITY_THRESHOLD)
        max_workers: 병렬 처리 스레드 수 (기본값: MAX_WORKERS)
    
    Returns:
        list: 유사한 상품 쌍 리스트
    """
    candidates = []
    
    # 비교 작업 리스트 생성 (썸네일이 있는 것만)
    comparison_tasks = []
    for p1 in products_coupang:
        for p2 in products_ssadagu:
            if p1.get("thumbnail_url") and p2.get("thumbnail_url"):
                comparison_tasks.append((p1, p2))
    
    total_comparisons = len(comparison_tasks)
    total_possible = len(products_coupang) * len(products_ssadagu)
    
    print(f"🔍 총 {len(products_coupang)}개(쿠팡) × {len(products_ssadagu)}개(싸다구) = {total_possible}개 비교")
    print(f"   (썸네일 있는 상품: {total_comparisons}개 비교)")
    print(f"⚡ 병렬 처리: 최대 {max_workers}개 스레드 사용\n")
    
    completed = 0
    start_time = time.time()
    
    # 병렬 처리로 비교 수행
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 각 비교 작업을 제출
        future_to_task = {
            executor.submit(calculate_combined_similarity, p1, p2): (p1, p2)
            for p1, p2 in comparison_tasks
        }
        
        # 완료된 작업 처리
        for future in as_completed(future_to_task):
            completed += 1
            p1, p2 = future_to_task[future]
            
            # 진행 상황 출력 (10%마다 또는 완료 시)
            if completed % max(1, total_comparisons // 10) == 0 or completed == total_comparisons:
                progress = (completed / total_comparisons) * 100
                elapsed = time.time() - start_time
                if completed > 0:
                    estimated_total = elapsed / (completed / total_comparisons)
                    remaining = max(0, estimated_total - elapsed)
                    print(f"  진행률: {progress:.1f}% ({completed}/{total_comparisons}) | "
                          f"경과: {elapsed/60:.1f}분 | 예상 남은 시간: {remaining/60:.1f}분")
            
            try:
                similarity = future.result()
                
                # 임계값 이상인 경우만 후보에 추가
                if similarity["combined_similarity"] >= threshold:
                    candidates.append({
                        "coupang_product": {
                            "title": p1.get("title", ""),
                            "price": p1.get("displayed_price") or p1.get("price") or p1.get("original_price", ""),
                            "thumbnail_url": p1.get("thumbnail_url", ""),
                            "product_link": p1.get("product_link", "")
                        },
                        "ssadagu_product": {
                            "title": p2.get("title", ""),
                            "price": p2.get("displayed_price") or p2.get("price") or p2.get("original_price", ""),
                            "thumbnail_url": p2.get("thumbnail_url", ""),
                            "product_link": p2.get("product_link", "")
                        },
                        "similarity": similarity
                    })
            except Exception as e:
                print(f"  ⚠️ 비교 실패: {str(e)[:50]}")
    
    elapsed_total = time.time() - start_time
    print(f"\n✅ 필터링 완료: {len(candidates)}개 후보 발견 (임계값: {threshold} 이상)")
    print(f"⏱️ 총 소요 시간: {elapsed_total/60:.1f}분")
    if elapsed_total > 0:
        print(f"📊 평균 처리 속도: {total_comparisons/(elapsed_total/60):.1f}개 비교/분\n")
    return candidates


print("✅ 종합 유사도 계산 및 필터링 함수 정의 완료")
print("   - calculate_combined_similarity(): 이미지 + 가격 + 상품명 + 상세정보 종합 유사도")
print("   - filter_similar_products(): 유사한 상품 쌍 필터링\n")


✅ 종합 유사도 계산 및 필터링 함수 정의 완료
   - calculate_combined_similarity(): 이미지 + 가격 + 상품명 + 상세정보 종합 유사도
   - filter_similar_products(): 유사한 상품 쌍 필터링



In [15]:
# Cell 8: 비교 및 Vision 실행 함수 정의

def parse_vision_response(raw_text: str):
    """```json 코드 블록을 포함한 Vision 응답을 안전하게 파싱합니다."""
    text = (raw_text or "").strip()
    if not text:
        return None
    code_block = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if code_block:
        text = code_block.group(1).strip()
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            parsed.setdefault("status", "ok")
        return parsed
    except json.JSONDecodeError:
        return None


def call_vision_api_with_retry(
    client,
    content,
    max_retries=VISION_MAX_RETRIES,
    base_wait=VISION_BASE_WAIT,
    rate_limit_wait=VISION_RATE_LIMIT_WAIT,
):
    """Vision API 호출을 재시도 로직과 함께 수행합니다."""
    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=VISION_MODEL,
                messages=content,
                max_tokens=400,
                timeout=40,
            )
            return response, None
        except AuthenticationError as e:
            return None, f"재시도 불가 인증 오류: {type(e).__name__}: {e}"
        except RateLimitError as e:
            wait_time = rate_limit_wait * (attempt + 1)
            print(f"  ⚠️ Rate limit 발생. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = f"Rate limit: {e}"
        except APIConnectionError as e:
            wait_time = base_wait ** (attempt + 1)
            print(f"  ⚠️ 네트워크 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = f"Connection error: {e}"
        except APIError as e:
            status = getattr(e, "status_code", None)
            if status and 400 <= status < 500:
                return None, f"재시도 불가 API 오류 (status {status}): {e}"
            wait_time = base_wait ** (attempt + 1)
            print(f"  ⚠️ API 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = f"API error: {e}"
        except Exception as e:
            if attempt == max_retries - 1:
                last_error = f"Unexpected error: {type(e).__name__}: {e}"
                break
            wait_time = base_wait ** (attempt + 1)
            print(f"  ⚠️ 일시적 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = str(e)
    return None, last_error or "Vision API 호출 실패"


def analyze_pair_with_vision(client, candidate, detail=VISION_DETAIL):
    """썸네일 2장을 Vision API로 비교하고 결과와 원문을 반환합니다."""
    coupang = candidate.get("coupang_product", {})
    ssadagu = candidate.get("ssadagu_product", {})
    coupang_thumb = coupang.get("thumbnail_url", "")
    ssadagu_thumb = ssadagu.get("thumbnail_url", "")

    if not coupang_thumb or not ssadagu_thumb:
        return {
            "status": "skipped",
            "reason": "missing_thumbnail",
            "message": "썸네일 URL이 없어 Vision 비교를 건너뜁니다.",
        }, None

    prompt = f"""두 쇼핑몰 썸네일이 같은 제품을 나타내는지 비교하세요. 반드시 JSON으로만 답변하세요.
출력 형식:
{{
  \"isSameProduct\": \"y\" 또는 \"n\",
  \"confidence\": 0-100 사이 숫자,
  \"keySimilarities\": ["항목"],
  \"keyDifferences\": ["항목"],
  \"verdict\": "간단한 판단 이유"
}}
제품 정보:
- Coupang: {coupang.get('title', 'N/A')} / 가격 {coupang.get('price', 'N/A')}
- Ssadagu: {ssadagu.get('title', 'N/A')} / 가격 {ssadagu.get('price', 'N/A')}"""

    content = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": coupang_thumb, "detail": detail}},
                {"type": "image_url", "image_url": {"url": ssadagu_thumb, "detail": detail}},
            ],
        }
    ]

    response, error_msg = call_vision_api_with_retry(client, content)
    if response is None:
        return {"status": "request_failed", "message": error_msg}, None

    raw_text = response.choices[0].message.content.strip()
    parsed = parse_vision_response(raw_text)
    if parsed is None:
        return {"status": "parse_error", "raw_response": raw_text}, raw_text
    return parsed, raw_text


def find_project_root(start_dir: Path, max_depth: int = 5) -> Path:
    """현재 경로에서 상위 디렉토리를 탐색하며 프로젝트 루트를 추정합니다."""
    current = start_dir
    for _ in range(max_depth):
        if (current / ".git").exists() or (current / "pyproject.toml").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    return start_dir


def run_compare_coupang_ssadagu(
    coupang_json_path,
    ssadagu_json_path,
    compare_output_path,
    vision_output_path,
    run_vision=True,
    top_k=VISION_TOP_K,
    vision_detail=VISION_DETAIL,
    max_workers=MAX_WORKERS,
):
    """쿠팡-싸다구 비교 파이프라인을 실행하고 결과를 저장합니다."""
    coupang_path = Path(coupang_json_path)
    ssadagu_path = Path(ssadagu_json_path)
    if not coupang_path.exists():
        raise FileNotFoundError(f"쿠팡 데이터 파일을 찾을 수 없습니다: {coupang_path}")
    if not ssadagu_path.exists():
        raise FileNotFoundError(f"싸다구 데이터 파일을 찾을 수 없습니다: {ssadagu_path}")

    print("📂 JSON 파일 로드 중...\n")
    try:
        with open(coupang_path, "r", encoding="utf-8") as f:
            coupang_data = json.load(f)
        with open(ssadagu_path, "r", encoding="utf-8") as f:
            ssadagu_data = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON 파싱 오류: {e}")

    coupang_products = coupang_data.get("products", [])
    ssadagu_products = ssadagu_data.get("products", [])

    print(f"✅ 쿠팡 상품: {len(coupang_products)}개")
    print(f"✅ 싸다구 상품: {len(ssadagu_products)}개")
    print(f"📊 검색 키워드: {coupang_data.get('search_keyword', 'N/A')}\n")

    print("=" * 60)
    similar_candidates = filter_similar_products(
        coupang_products,
        ssadagu_products,
        threshold=SIMILARITY_THRESHOLD,
        max_workers=max_workers,
    )
    print("=" * 60)

    compare_output = Path(compare_output_path)
    compare_output.parent.mkdir(parents=True, exist_ok=True)
    result_payload = {
        "search_keyword": coupang_data.get("search_keyword", ""),
        "comparison_date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "settings": {
            "similarity_threshold": SIMILARITY_THRESHOLD,
            "image_weight": IMAGE_WEIGHT,
            "price_weight": PRICE_WEIGHT,
            "title_weight": TITLE_WEIGHT,
            "ocr_specs_weight": OCR_SPECS_WEIGHT,
        },
        "statistics": {
            "total_coupang_products": len(coupang_products),
            "total_ssadagu_products": len(ssadagu_products),
            "total_comparisons": len(coupang_products) * len(ssadagu_products),
            "candidates_found": len(similar_candidates),
        },
        "candidates": similar_candidates,
    }

    with open(compare_output, "w", encoding="utf-8") as f:
        json.dump(result_payload, f, ensure_ascii=False, indent=2)

    print(f"✅ 결과 저장 완료: {compare_output}")
    print("📊 비교 결과 요약:")
    print(f"   - 쿠팡 상품 수: {len(coupang_products)}개")
    print(f"   - 싸다구 상품 수: {len(ssadagu_products)}개")
    print(f"   - 총 비교 횟수: {len(coupang_products) * len(ssadagu_products)}회")
    print(f"   - 유사한 상품 쌍: {len(similar_candidates)}개 (임계값: {SIMILARITY_THRESHOLD})\n")

    vision_results = None
    if not run_vision:
        print("⚠️ Vision 비교가 비활성화되어 있어 이미지 정밀 비교를 건너뜁니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    if not similar_candidates:
        print("⚠️ Vision 비교를 수행할 후보가 없습니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    project_root = find_project_root(Path.cwd())
    env_path = project_root / ".env"
    if not env_path.exists():
        print(f"⚠️ .env 파일을 찾을 수 없어 Vision 비교를 건너뜁니다: {env_path}")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    load_dotenv(env_path)
    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        print("⚠️ OPENAI_API_KEY가 설정되어 있지 않아 Vision 비교를 건너뜁니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }
    if not (api_key.startswith("sk-") or api_key.startswith("sk-proj-")):
        print("⚠️ OPENAI_API_KEY 형식이 현재 지원되는 접두사(sk-, sk-proj-)와 일치하지 않아 Vision 비교를 건너뜁니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    client = OpenAI(api_key=api_key)
    top_candidates = sorted(
        similar_candidates,
        key=lambda x: x["similarity"]["combined_similarity"],
        reverse=True,
    )[:max(1, top_k)]

    if not top_candidates:
        print("⚠️ Vision 비교를 수행할 후보가 없습니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    print(f"🔎 Vision API 정밀 비교 시작 (상위 {len(top_candidates)}개 후보, detail='{vision_detail}')\n")
    vision_results = []
    for idx, candidate in enumerate(top_candidates, 1):
        coupang = candidate["coupang_product"]
        ssadagu = candidate["ssadagu_product"]
        combined = candidate["similarity"]["combined_similarity"]
        print("=" * 70)
        print(f"[{idx}] 종합 유사도 {combined:.2%}")
        print(f"  Coupang : {coupang.get('title', '')[:80]}...")
        print(f"             가격 {coupang.get('price')}")
        print(f"  Ssadagu : {ssadagu.get('title', '')[:80]}...")
        print(f"             가격 {ssadagu.get('price')}")

        result, raw_text = analyze_pair_with_vision(client, candidate, detail=vision_detail)
        vision_results.append({
            "candidate": candidate,
            "vision_result": result,
            "raw_text": raw_text,
        })

        status = result.get("status")
        if status == "skipped":
            print(f"  ⚠️ Vision 비교 스킵: {result.get('message')}")
        elif status == "request_failed":
            print(f"  ⚠️ Vision API 호출 실패: {result.get('message')}")
        elif status == "parse_error":
            preview = (result.get("raw_response") or "")[:200]
            print("  ⚠️ JSON 파싱 실패. 원문 미리보기:")
            print(f"     {preview}...")
        else:
            print("  ✅ Vision 결과:")
            print(f"     동일 여부 : {result.get('isSameProduct', 'N/A')}, 신뢰도 {result.get('confidence', 'N/A')}%")
            similarities = result.get("keySimilarities") or []
            if similarities:
                print(f"     공통점   : {', '.join(similarities[:3])}")
            differences = result.get("keyDifferences") or []
            if differences:
                print(f"     차이점   : {', '.join(differences[:3])}")
            if result.get("verdict"):
                print(f"     판단 사유: {result['verdict']}")

    vision_output = Path(vision_output_path)
    vision_output.parent.mkdir(parents=True, exist_ok=True)
    export_payload = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model": VISION_MODEL,
        "detail": vision_detail,
        "top_k": len(top_candidates),
        "results": [
            {
                "coupang_product": item["candidate"]["coupang_product"],
                "ssadagu_product": item["candidate"]["ssadagu_product"],
                "similarity": item["candidate"]["similarity"],
                "vision_result": item["vision_result"],
            }
            for item in vision_results
        ],
    }
    with open(vision_output, "w", encoding="utf-8") as f:
        json.dump(export_payload, f, ensure_ascii=False, indent=2)

    print("\n🎯 Vision 비교 완료. 결과가 저장되었습니다.")
    print(f"📝 저장 경로: {vision_output}")

    return {
        "coupang_products": coupang_products,
        "ssadagu_products": ssadagu_products,
        "similar_candidates": similar_candidates,
        "vision_results": vision_results,
    }


In [16]:
# Cell 7: 실행 스크립트 생성 및 실행

import os
import sys
import tempfile
import subprocess
import inspect
import textwrap

COUPANG_JSON_PATH = Path("../crawling_tests/coupang_search_results.json").resolve()
SSADAGU_JSON_PATH = Path("../crawling_tests/ssadagu_search_results.json").resolve()
COMPARE_OUTPUT_JSON = Path("compare_coupang_ssadagu_results.json").resolve()
VISION_OUTPUT_JSON = Path("vision_top3_results.json").resolve()
RUN_VISION = True
VISION_TOP_K_OVERRIDE = VISION_TOP_K
VISION_DETAIL_LEVEL = VISION_DETAIL
MAX_WORKERS_OVERRIDE = MAX_WORKERS

print("▶ 쿠팡 vs 싸다구 상품 비교를 시작합니다...\n")

function_list = [
    get_image_hash,
    calculate_image_similarity,
    remove_marketing_words,
    extract_spec_keywords,
    calculate_title_similarity,
    parse_price,
    calculate_price_similarity,
    normalize_keyword,
    convert_specs_to_text,
    clean_ocr_text,
    calculate_ocr_specs_similarity,
    calculate_combined_similarity,
    filter_similar_products,
    parse_vision_response,
    call_vision_api_with_retry,
    analyze_pair_with_vision,
    find_project_root,
    run_compare_coupang_ssadagu,
]

function_sources = [textwrap.dedent(inspect.getsource(fn)) for fn in function_list]
functions_code = "\n\n".join(function_sources)

coupang_literal = repr(str(COUPANG_JSON_PATH))
ssadagu_literal = repr(str(SSADAGU_JSON_PATH))
compare_literal = repr(str(COMPARE_OUTPUT_JSON))
vision_literal = repr(str(VISION_OUTPUT_JSON))

script_lines = [
    "import os",
    "import json",
    "import time",
    "import re",
    "from pathlib import Path",
    "from io import BytesIO",
    "from concurrent.futures import ThreadPoolExecutor, as_completed",
    "from difflib import SequenceMatcher",
    "import requests",
    "import imagehash",
    "from PIL import Image",
    "from dotenv import load_dotenv",
    "from openai import OpenAI, RateLimitError, APIError, APIConnectionError, AuthenticationError",
    "",
    f"SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}",
    f"IMAGE_WEIGHT = {IMAGE_WEIGHT}",
    f"PRICE_WEIGHT = {PRICE_WEIGHT}",
    f"TITLE_WEIGHT = {TITLE_WEIGHT}",
    f"OCR_SPECS_WEIGHT = {OCR_SPECS_WEIGHT}",
    f"MAX_WORKERS = {MAX_WORKERS_OVERRIDE}",
    "TOTAL_WEIGHT = IMAGE_WEIGHT + PRICE_WEIGHT + TITLE_WEIGHT + OCR_SPECS_WEIGHT",
    "",
    "_image_hash_cache = {}",
    "",
    f"VISION_MODEL = {VISION_MODEL!r}",
    f"VISION_DETAIL = {VISION_DETAIL_LEVEL!r}",
    f"VISION_TOP_K = {VISION_TOP_K_OVERRIDE}",
    f"VISION_MAX_RETRIES = {VISION_MAX_RETRIES}",
    f"VISION_BASE_WAIT = {VISION_BASE_WAIT}",
    f"VISION_RATE_LIMIT_WAIT = {VISION_RATE_LIMIT_WAIT}",
    "",
    functions_code,
    "",
    'if __name__ == "__main__":',
    f"    run_compare_coupang_ssadagu(",
    f"        coupang_json_path={coupang_literal},",
    f"        ssadagu_json_path={ssadagu_literal},",
    f"        compare_output_path={compare_literal},",
    f"        vision_output_path={vision_literal},",
    f"        run_vision={RUN_VISION},",
    f"        top_k={VISION_TOP_K_OVERRIDE},",
    f"        vision_detail={VISION_DETAIL_LEVEL!r},",
    f"        max_workers={MAX_WORKERS_OVERRIDE},",
    "    )",
    '    print("\\n✅ 비교 스크립트 실행 완료")',
]

script_content = "\n".join(script_lines)

fd, script_path = tempfile.mkstemp(suffix="_compare_coupang_ssadagu.py", text=True)
os.close(fd)
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)

try:
    # Jupyter Notebook 호환: subprocess.Popen으로 실시간 출력 처리
    # OS 독립적인 인코딩 처리를 위해 환경 변수 설정
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'  # Python 출력 인코딩을 UTF-8로 강제
    
    # subprocess.Popen으로 프로세스 시작
    process = subprocess.Popen(
        [sys.executable, script_path],
        stdout=subprocess.PIPE,  # 파이프로 출력 캡처
        stderr=subprocess.STDOUT,  # stderr를 stdout에 병합
        text=True,
        encoding='utf-8',  # UTF-8 인코딩 명시 (Windows/Mac/Linux 모두 지원)
        errors='replace',  # 디코딩 에러 시 대체 문자 사용
        env=env,  # 환경 변수 전달 (OS 독립적)
        bufsize=1,  # 라인 버퍼링 (실시간 출력)
    )
    
    # 실시간으로 출력 읽기 및 표시
    for line in process.stdout:
        print(line, end='', flush=True)  # Jupyter Notebook에 실시간 출력
    
    # 프로세스 종료 대기
    returncode = process.wait()
    
    if returncode != 0:
        print("\n⚠️ 프로세스가 에러로 종료되었습니다.")
finally:
    try:
        os.remove(script_path)
    except OSError:
        pass


▶ 쿠팡 vs 싸다구 상품 비교를 시작합니다...

📂 JSON 파일 로드 중...

✅ 쿠팡 상품: 30개
✅ 싸다구 상품: 30개
📊 검색 키워드: 컴퓨터

🔍 총 30개(쿠팡) × 30개(싸다구) = 900개 비교
   (썸네일 있는 상품: 900개 비교)
⚡ 병렬 처리: 최대 10개 스레드 사용

  진행률: 10.0% (90/900) | 경과: 1.5분 | 예상 남은 시간: 13.2분
  진행률: 20.0% (180/900) | 경과: 1.9분 | 예상 남은 시간: 7.6분
  진행률: 30.0% (270/900) | 경과: 2.4분 | 예상 남은 시간: 5.6분
  진행률: 40.0% (360/900) | 경과: 2.8분 | 예상 남은 시간: 4.3분
  진행률: 50.0% (450/900) | 경과: 3.4분 | 예상 남은 시간: 3.4분
  진행률: 60.0% (540/900) | 경과: 3.9분 | 예상 남은 시간: 2.6분
  진행률: 70.0% (630/900) | 경과: 4.3분 | 예상 남은 시간: 1.8분
  진행률: 80.0% (720/900) | 경과: 4.8분 | 예상 남은 시간: 1.2분
  진행률: 90.0% (810/900) | 경과: 5.1분 | 예상 남은 시간: 0.6분
  진행률: 100.0% (900/900) | 경과: 5.7분 | 예상 남은 시간: 0.0분

✅ 필터링 완료: 5개 후보 발견 (임계값: 0.4 이상)
⏱️ 총 소요 시간: 5.7분
📊 평균 처리 속도: 158.7개 비교/분

✅ 결과 저장 완료: C:\projects\Final-AI-Fork\dev\compare_test\compare_coupang_ssadagu_results.json
📊 비교 결과 요약:
   - 쿠팡 상품 수: 30개
   - 싸다구 상품 수: 30개
   - 총 비교 횟수: 900회
   - 유사한 상품 쌍: 5개 (임계값: 0.4)

🔎 Vision API 정밀 비교 시작 (상위 3개 후보, detail='low')

[1] 